# Exercise 2.4: DLESyM Seasonal Forecast (90 Days)

**Level**: Intermediate | **Time**: 45 min | **GPU**: Required (V100+)

Run a subseasonal-to-seasonal (S2S) forecast with DLESyM — NVIDIA's coupled atmosphere-ocean model. Predict beyond the 2-week limit where deterministic weather forecasting breaks down.

---

**Why S2S matters:**
- Agriculture: planting decisions need 4-8 week outlooks
- Energy: heating/cooling demand planning
- Water: reservoir operations need seasonal inflow forecasts
- Insurance: catastrophe modeling needs multi-month perspectives

In [ ]:
# Install Earth2Studio + dependencies (don't install torch — Colab has it)
!pip install -q "earth2studio>=0.13.0" torch-harmonics matplotlib xarray zarr scipy
!pip install -q "makani @ git+https://github.com/NVIDIA/makani.git"

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE — need GPU'}")

## Part 1: Run 90-Day Forecast

DLESyM differs from FCN3:
- Uses **ERA5** initial conditions (via ARCO), not GFS
- Uses **daily** timesteps (not 6-hourly)
- Jointly predicts **atmosphere + ocean** (coupled model)

In [ ]:
from earth2studio.models.px import DLESyMLatLon
from earth2studio.data import ARCO
from earth2studio.io import ZarrBackend
from earth2studio import run

print("Loading DLESyM model...")
model = DLESyMLatLon.load_model(DLESyMLatLon.load_default_package())

print("Running 90-day forecast...")
io = run.deterministic(
    time=["2025-01-01T00:00:00"],
    nsteps=90,          # 90 days (~13 weeks)
    model=model,
    data=ARCO(),        # ERA5 reanalysis
    io=ZarrBackend("dlesym_90day.zarr")
)
print("Done!")

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

ds = xr.open_zarr("dlesym_90day.zarr")
lats = ds.coords['lat'].values
lons = ds.coords['lon'].values
print(f"Dims: {dict(ds.dims)}")
print(f"Variables: {list(ds.coords['variable'].values)}")

## Part 2: Weekly Temperature Evolution

In [ ]:
n_steps = len(ds.coords['lead_time'])
n_weeks = min(n_steps // 7, 12)

fig, axes = plt.subplots(3, 4, figsize=(20, 14))
axes = axes.flatten()

for week in range(n_weeks):
    s_start, s_end = week * 7, min((week + 1) * 7, n_steps)
    weekly = ds.sel(variable="t2m").isel(
        time=0, lead_time=slice(s_start, s_end)
    ).mean(dim="lead_time").values.squeeze() - 273.15
    
    axes[week].contourf(lons, lats, weekly, levels=np.arange(-40, 45, 5), cmap='RdYlBu_r')
    axes[week].set_title(f'Week {week+1} (Days {s_start+1}-{s_end})')

for j in range(n_weeks, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('DLESyM Weekly Mean Temperature (°C) — 90-Day Forecast', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

## Part 3: Regional Time Series

Compare how different climate zones evolve over 3 months.

In [ ]:
regions = {
    "Tropics (10°S-10°N)": (-10, 10),
    "NH Midlats (30-60°N)": (30, 60),
    "SH Midlats (30-60°S)": (-60, -30),
    "Arctic (>60°N)": (60, 90),
}

t2m = ds.sel(variable="t2m").isel(time=0).values.squeeze() - 273.15

fig, ax = plt.subplots(figsize=(12, 6))
for name, (lat_min, lat_max) in regions.items():
    mask = (lats >= lat_min) & (lats <= lat_max)
    regional = [np.nanmean(t2m[s][mask, :]) for s in range(t2m.shape[0])]
    ax.plot(range(len(regional)), regional, linewidth=2, label=name)

ax.set_xlabel('Forecast Day')
ax.set_ylabel('Temperature (°C)')
ax.set_title('DLESyM Regional Temperature Evolution', fontsize=14)
ax.legend()
ax.grid(True, alpha=0.3)

for d, lbl in [(14, 'Wk 2'), (28, 'Wk 4'), (56, 'Wk 8')]:
    ax.axvline(x=d, color='gray', ls='--', alpha=0.4)
    ax.text(d, ax.get_ylim()[1], f' {lbl}', va='top', fontsize=9, color='gray')

plt.tight_layout()
plt.show()

## Part 4: Week 3-4 Anomaly Outlook

Standard S2S outlook format: anomaly from baseline.

In [ ]:
# Week 3-4 mean vs Day 1 baseline
future = ds.sel(variable="t2m").isel(time=0, lead_time=slice(14, 28)).mean(dim="lead_time").values.squeeze()
baseline = ds.sel(variable="t2m").isel(time=0, lead_time=0).values.squeeze()
anomaly = future - baseline  # K difference = °C difference

fig, ax = plt.subplots(figsize=(14, 6))
cf = ax.contourf(lons, lats, anomaly, levels=np.arange(-10, 11, 1), cmap='RdBu_r')
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
ax.set_title('Temperature Anomaly — Weeks 3-4 vs Day 1\nDLESyM | Red=Warmer, Blue=Cooler', fontsize=14)
plt.colorbar(cf, ax=ax, label='Anomaly (°C)')
plt.tight_layout()
plt.show()

---

## Exercises

1. Compare Week 1-2 anomaly vs Week 5-6 — does the pattern persist or change?
2. Run a DLESyM ensemble (20 members) — how does S2S spread compare to FCN3?
3. If DLESyM outputs SST, plot ocean temperature evolution over 90 days
4. Pick a monsoon region (India, West Africa) — can you see seasonal onset?
5. Compare against CPC seasonal outlook: https://www.cpc.ncep.noaa.gov/